In [ ]:
import pandas as pd
def data_split(file_path:str, label_col:str, train_ratio=0.8):
    df_raw = pd.read_excel(file_path)
    df_raw.set_index('date', inplace=True)

    # 先删除目标列为空的行
    df_raw = df_raw.dropna(subset=[label_col])

    # 涨跌标签：下一期比当前期涨为 1，跌为 -1，不变为 0，不同业务可更换标签
    df_raw['label'] = (df_raw[label_col].shift(-1) - df_raw[label_col]).apply(
        lambda x: -1 if x > 0 else (1 if x < 0 else 0)
    )

    # 若为 0，则继承前一个标签
    df_raw['label'] = df_raw['label'].replace(0, method='ffill')

    # 删除最后一行（因为没有未来数据做标签）
    df_clean = df_raw.iloc[:-1]

    # 假设 df_clean 已经准备好，索引为时间或顺序
    label_col = "label"
    split_point = int(len(df_clean) * 0.8)

    # 按顺序切分
    df_train = df_clean.iloc[:split_point]
    df_test  = df_clean.iloc[split_point:]

    # 拆分特征和标签
    X_train, y_train = df_train.drop(columns=[label_col]), df_train[label_col]
    X_test,  y_test  = df_test.drop(columns=[label_col]),  df_test[label_col]

    train_data = pd.concat([X_train, y_train], axis=1)
    test_data = pd.concat([X_test, y_test], axis=1)
    return train_data, test_data

## 训练模型

In [ ]:
label_col = '5Y'
train_data, test_data = data_split('merged_tier2.xlsx', label_col)
from autogluon.tabular import TabularDataset, TabularPredictor
predictor = TabularPredictor(label=label_col).fit(train_data,presets='best')

## 评估模型，显示准确率等

In [ ]:
predictor.evaluate(test_data,auxiliary_metrics=True)

In [ ]:
# 模型保存路径
predictor.path

In [ ]:
from autogluon.tabular import TabularPredictor
import pandas as pd

# 加载模型并预测
# predictor = TabularPredictor.load("/root/AutogluonModels/ag-20251017_002741")
predictions = predictor.predict(test_data)

# 构建结果 DataFrame
df_result = test_data.reset_index()[['date', label_col, 'label']].copy()
df_result['predictions'] = predictions.values

# 保存结果
output_file = 'prediction_results_国开债十年.xlsx'
df_result.to_excel(output_file, index=False)

print(f"预测结果已保存到 {output_file}")